# P09C — Symbolic regression dan mapping function

Jalankan P09A dahulu. Pencarian bentuk persamaan terbatas ini bukan algoritme evolutionary symbolic regression penuh. Tujuannya membandingkan kandidat mekanika dengan persamaan deskriptif serta menguji ekstrapolasi.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score
b=pd.read_csv('Bab9_Paket/Data/beam_moment_synthetic.csv')
# Train on lower-to-middle steel grades; evaluate interpolation and higher-grade extrapolation separately.
train=b[b.fy_mpa<=410]; test_in=b[(b.fy_mpa>300)&(b.fy_mpa<=410)].sample(frac=.35,random_state=19); train=train.drop(test_in.index); test_ex=b[b.fy_mpa>440]
def eval_formula(cols,label):
 m=LinearRegression().fit(train[cols],train.moment_kNm)
 for name,t in [('train-domain holdout',test_in),('higher-grade extrapolation',test_ex)]:
  pred=m.predict(t[cols]); print(label,name,'MAE=',round(mean_absolute_error(t.moment_kNm,pred),2),'R2=',round(r2_score(t.moment_kNm,pred),4))
 print(label,'coefficients=',dict(zip(cols,m.coef_.round(6))),'intercept=',round(m.intercept_,4)); return m
train=train.copy(); test_in=test_in.copy(); test_ex=test_ex.copy()
for d in [train,test_in,test_ex]: d['Z_times_fy']=d.Z_cm3*d.fy_mpa
m_phys=eval_formula(['Z_times_fy'],'mechanics candidate: 0.001 Z fy')
m_add=eval_formula(['Z_cm3','fy_mpa'],'additive descriptive candidate')
rf=RandomForestRegressor(n_estimators=200,min_samples_leaf=3,random_state=42,n_jobs=1).fit(train[['Z_cm3','fy_mpa']],train.moment_kNm)
for label,t in [('in-range',test_in),('high-fy',test_ex)]: print('RF',label,'MAE=',round(mean_absolute_error(t.moment_kNm,rf.predict(t[['Z_cm3','fy_mpa']])),2))
print('DGP ground truth: moment_kNm = 0.001 * Z_cm3 * fy_mpa before 1.2% noise. Candidate formula is interpretable but only validated within stated ranges.')